# LLM Satisfaction Predictor — Exploratory Data Analysis

This notebook explores the `genai_llm_usage_dataset_1000.csv` dataset to:
- Understand what columns exist and what they mean
- Identify the **target variable** (what we want to predict)
- Detect **data quality issues** (missing values, duplicates, outliers)
- Decide **which features to use** and which to exclude (leakage prevention)
- Understand the **distribution** of the satisfaction score

**This is an exploration notebook. Production code lives in `src/`.**

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='darkgrid')

print('Libraries loaded.')

## 2. Load Dataset

In [ ]:
# The path is relative to the project root.
# Run this notebook from the project root directory.
DATA_PATH = '../data/raw/genai_llm_usage_dataset_1000.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns')

## 3. Display Shape and First Rows

In [ ]:
df.head()

## 4. Column Names

In [ ]:
print('Columns:')
for col in df.columns:
    print(f'  - {col}')

## 5. Data Types

In [ ]:
df.dtypes

## 6. Missing Values

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

## 7. Duplicate Rows

In [ ]:
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes}')

## 8. Summary Statistics

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

## 9. Target Variable: user_satisfaction

**This is what we want to predict.**

It represents how satisfied the user was with the LLM response.

This is a **regression** problem because we want to predict a continuous score (even though the actual values are 3, 4, or 5).

In [ ]:
print('user_satisfaction unique values:', sorted(df['user_satisfaction'].unique()))
print()
print('Value counts:')
print(df['user_satisfaction'].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(7, 4))
df['user_satisfaction'].value_counts().sort_index().plot(kind='bar', color=['#ef4444', '#4F8EF7', '#22c55e'])
plt.title('Distribution of User Satisfaction Scores')
plt.xlabel('Satisfaction Score')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Categorical Feature Analysis

In [ ]:
cat_cols = ['model_name', 'application_domain', 'task_type']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, cat_cols):
    df[col].value_counts().plot(kind='bar', ax=ax, color='#4F8EF7')
    ax.set_title(col)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Average satisfaction by model
print('Average satisfaction by model:')
print(df.groupby('model_name')['user_satisfaction'].mean().sort_values(ascending=False).round(3))

In [ ]:
# Average satisfaction by domain
print('Average satisfaction by application domain:')
print(df.groupby('application_domain')['user_satisfaction'].mean().sort_values(ascending=False).round(3))

## 11. Numerical Feature Analysis

In [ ]:
num_cols = ['prompt_length', 'total_tokens', 'temperature', 'top_p', 'latency_sec', 'estimated_cost_usd']
df[num_cols].hist(bins=30, figsize=(14, 8), color='#4F8EF7', edgecolor='white', linewidth=0.4)
plt.suptitle('Numerical Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

## 12. Correlation Analysis

In [ ]:
plt.figure(figsize=(10, 7))
corr = df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 13. Outlier Inspection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['prompt_length', 'temperature', 'top_p']):
    ax.boxplot(df[col])
    ax.set_title(col)
plt.suptitle('Outlier Check (Box Plots)')
plt.tight_layout()
plt.show()

## 14. Target Leakage Analysis

**Target leakage** means using information in your model that would not be available at prediction time.

### Excluded columns and reasons:

| Column | Reason for Exclusion |
|--------|---------------------|
| `session_id` | Row identifier — no predictive signal |
| `total_tokens` | Includes completion tokens — only known AFTER the LLM responds |
| `latency_sec` | Response time — only measured AFTER the LLM responds |
| `hallucination_flag` | Only evaluated AFTER reading the response |
| `successful_response` | Only known AFTER the response arrives |
| `estimated_cost_usd` | Fully calculated AFTER usage |
| `user_satisfaction` | This IS the target — never an input feature |

### Selected features (all known BEFORE or DURING request setup):
- `model_name`, `application_domain`, `task_type` — chosen before the request
- `prompt_length` — known when the prompt is written
- `temperature`, `top_p` — model parameters set before calling the API
- `rag_enabled` — configuration decision made before the request

In [ ]:
# Verify total_tokens is post-response (includes completion tokens)
df['output_tokens_estimate'] = df['total_tokens'] - df['prompt_length']
print('Delta (total_tokens - prompt_length) statistics:')
print(df['output_tokens_estimate'].describe())
print('\nThis varies from 21 to 1200, confirming total_tokens includes output tokens (POST-RESPONSE).')
df.drop(columns=['output_tokens_estimate'], inplace=True)

## 15. Final Feature Selection Summary

The production model (`src/train.py`) uses these 7 features:

```
NUMERIC (4):     prompt_length, temperature, top_p, rag_enabled
CATEGORICAL (3): model_name, application_domain, task_type

TARGET: user_satisfaction (values: 3, 4, 5)
```

### Why regression and not classification?

Although the target only takes 3 discrete values (3, 4, 5), we treat this as **regression** because:
1. The values are **ordered** — 4 is better than 3, 5 is better than 4
2. Regression allows the model to predict intermediate values (e.g., 4.2) which gives richer information
3. Evaluation metrics like MAE give intuitive interpretations (e.g., off by 0.4 satisfaction points)

In [ ]:
# Final feature set
FEATURES = ['prompt_length', 'temperature', 'top_p', 'rag_enabled',
            'model_name', 'application_domain', 'task_type']
TARGET = 'user_satisfaction'

X = df[FEATURES]
y = df[TARGET]

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape:         {y.shape}')
print(f'Target range:         [{y.min()}, {y.max()}]')
print()
print('Ready for modelling. See src/train.py for the production pipeline.')